# Parte 1 — Engenharia de Dados: Curadoria e Qualidade na Silver

Aplica os tratamentos das sete tabelas Silver, ajusta os tipos e usa nomes de colunas em português. As tabelas Bronze são usadas como origem. As gravações na Silver usam Delta Overwrite.


## Configuração da camada Silver

Define os schemas de origem e destino com o mesmo catálogo usado nos notebooks anteriores.


In [0]:
# Configura os nomes das camadas para a leitura e a gravação das tabelas.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

## Apoio — Funções de checagem de qualidade

dq_check conta as linhas que não atendem à condição. dq_check_unique conta as chaves duplicadas. As funções mostram PASS ou FAIL e guardam o resultado das verificações.


In [0]:
# As funções abaixo registram os resultados das checagens de qualidade.

from datetime import datetime
from pyspark.sql import Row, functions as F

dq_results = []

def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=failed,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")
    return passed


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    passed = duplicates == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=duplicates,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")
    return passed

## 1. silver.tb_info_filmes — Tratamento dos metadados

Mantém a versão mais recente por id de filme usando ingestion_datetime. Converte as datas em três formatos, normaliza e traduz o status, renomeia as colunas e deriva ano_lancamento. Valores não mapeáveis de status ficam como Não Informado.


In [0]:
# O timestamp é o primeiro campo do struct, por isso define a versão mais recente.

from pyspark.sql import functions as F

df_bronze_info_filmes = spark.table(f"{bronze_schema}.tb_movies_info")

df_silver_info_filmes = (
    df_bronze_info_filmes

    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .withColumn("ingestion_datetime", F.col("ingestion_datetime").cast("timestamp"))

    # Agrupa por filme e seleciona o registro mais recente
    .groupBy("id")
    .agg(
        F.max(
            F.struct(
                "ingestion_datetime",
                "title",
                "original_title",
                "release_date",
                "runtime",
                "original_language",
                "status",
                "overview",
                "tagline"
            )
        ).alias("mais_recente")
    )
    .select("id", "mais_recente.*")

    # Tipagem e conversão das datas
    .withColumn("runtime", F.expr("try_cast(runtime AS INT)"))
    .withColumn("release_date", F.trim(F.col("release_date").cast("string")))
    .withColumn(
        "release_date",
        F.coalesce(
            F.try_to_timestamp(F.col("release_date"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("release_date"), F.lit("dd/MM/yyyy")),
            F.try_to_timestamp(F.col("release_date"), F.lit("MM-dd-yyyy"))
        ).cast("date")
    )

    # Normalização do status
    .withColumn(
        "status",
        F.trim(
            F.regexp_replace(
                F.regexp_replace(F.lower("status"), r"[-_‐‑‒–—−]+", " "),
                r"\s+", " "
            )
        )
    )

    # Tradução do status
    .withColumn(
        "status",
        F.when(F.col("status") == "released", "Lançado")
         .when(F.col("status") == "post production", "Pós-Produção")
         .when(F.col("status") == "in production", "Em Produção")
         .when(F.col("status") == "planned", "Planejado")
         .when(F.col("status") == "rumored", "Rumores")
         .when(F.col("status") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )

    .select(
        "id", "title", "original_title", "release_date",
        "runtime", "original_language", "status", "overview", "tagline"
    )

    # Mapeamento das colunas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    # Coluna derivada
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

dq_check(
    "silver.tb_info_filmes", "id_filme não nulo",
    df_silver_info_filmes, F.col("id_filme").isNotNull()
)

dq_check_unique(
    "silver.tb_info_filmes", "id_filme único",
    df_silver_info_filmes, ["id_filme"]
)

display(df_silver_info_filmes.limit(10))
df_silver_info_filmes.printSchema()

[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
14564,Rings,Rings,2017-02-01,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die.",null,2017
45033,20 Seconds of Joy,20 Seconds of Joy,2018-01-01,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear.",null,2018
47933,Independence Day: Resurgence,Independence Day: Resurgence,2016-06-22,120,en,Lançado,"We always knew they were coming back. Using recovered alien technology, the nations of Earth have collaborated on an immense defense program to protect the planet. But nothing can prepare us for the aliens’ advanced and unprecedented force. Only the ingenuity of a few brave men and women can bring our world back from the brink of extinction.",We had twenty years to prepare. So did they.,2016
68735,Warcraft,Warcraft,2016-05-25,123,en,Lançado,"The peaceful realm of Azeroth stands on the brink of war as its civilization faces a fearsome race of invaders: orc warriors fleeing their dying home to colonize another. As a portal opens to connect the two worlds, one army faces destruction and the other faces extinction. From opposing sides, two heroes are set on a collision course that will decide the fate of their family, their people, and their home.",Two worlds. One home.,2016
79759,The Imaginary Invalid,Le malade imaginaire,2019-11-13,0,fr,Lançado,null,null,2019
86822,Voyage of Time: Life's Journey,Voyage of Time: Life's Journey,2017-03-10,90,en,Lançado,"A celebration of the universe, displaying the whole of time, from its start to its final collapse. This film examines all that occurred to prepare the world that stands before us now: science and spirit, birth and death, the grand cosmos and the minute life systems of our planet. (Wide release version with narration by Cate Blanchett.)",From So Simple a Beginning.,2017
98547,One Fall,One Fall,2016-09-09,90,en,Lançado,"Set in the rustic Midwestern town of One Fall, the film tells the story of a man, James (Marcus Dean Fuller), who miraculously survived a horrific fall from a spectacular 200 foot-high cliff and was never heard from again. However, three years after vanishing James chooses to return to his hometown of One Fall -- but he returns a changed man. For an incomprehensible reason James has developed supernatural healing abilities since the fall. He must decide whether to use his abilities to help the ones he once turned his back on, or to continue running from his mysterious past.",You can't choose your destiny... It chooses you.,2016
115436,Hason Raja,Hason Raja,2017-03-31,165,bn,Lançado,"The story of Hason Raja, a flamboyant, ruthless zamindar from Sylhet who fell in love with Dilaram, who transformed him and later he became a poet, singing songs and wandering all across Bangladesh.",null,2017
137116,Smurfs: The Lost Village,Smurfs: The Lost Village,2017-03-23,89,en,Lançado,"In this fully animated, all-new take on the Smurfs, a mysterious map sets Smurfette and her friends Brainy, Clumsy and Hefty on an exciting race through the Forbidden Forest leading to the discovery of the biggest secret in Smurf history.",null,2017
207680,Return to... Return to Nuke 'Em High aka Vol. 2,Return to... Return to Nuke 'Em High aka Vol. 2,2017-01-15,72,en,Lançado,"Following the events of Volume 1, the mutated glee club continue their violent rampage in Tromaville. Chrissy and Lauren, two innocent lesbian lovers, must fight not only the Cretins, mutants, and monsters but also the evil Tromorganic Foodstuffs Conglomerate.

root
 |-- id_filme: long (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)



## 1. silver.tb_info_filmes — Gravação

Grava os metadados tratados em Delta. Overwrite substitui o conteúdo da tabela Silver, e overwriteSchema aplica o esquema resultante.


In [0]:
# Grava a tabela de informações dos filmes com os tipos e nomes tratados.

df_silver_info_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

## 2. silver.tb_financeiro_filmes — Seleção da cotação

Seleciona a cotação de compra mais recente com data válida e valor positivo. Essa taxa será aplicada na conversão de dólar para real.


In [0]:
# Usa a cotação de compra mais recente disponível na Bronze.

cotacao = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .withColumn("dataHoraCotacao", F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)"))
    .withColumn("cotacaoCompra", F.expr("try_cast(cotacaoCompra AS DECIMAL(18,6))"))
    .filter(
        F.col("dataHoraCotacao").isNotNull()
        & (F.col("cotacaoCompra") > 0)
    )
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()
)

if cotacao is None:
    raise ValueError("Não há cotação válida na tabela Bronze.")

taxa_dolar = cotacao["cotacaoCompra"]
print(f"Cotação utilizada: R$ {taxa_dolar} por dólar")

Cotação utilizada: R$ 4.992900 por dólar


## 2. silver.tb_financeiro_filmes — Limpeza e cálculos

Transforma textos de ausência em NULL, remove símbolos de moeda e separadores e converte orçamento e receita para decimal. Valores zerados ou negativos ficam ausentes. Calcula BRL, lucro e margem percentual usando lucro dividido por receita, com proteção contra divisão por zero.


In [0]:
# Os cálculos dependentes permanecem NULL quando faltar orçamento ou receita.

from pyspark.sql import functions as F

# 1. Cotação de compra mais recente da Bronze
cotacao = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .withColumn("dataHoraCotacao", F.expr("try_cast(dataHoraCotacao AS TIMESTAMP)"))
    .withColumn("cotacaoCompra", F.expr("try_cast(cotacaoCompra AS DECIMAL(18,6))"))
    .filter(
        F.col("dataHoraCotacao").isNotNull()
        & (F.col("cotacaoCompra") > 0)
    )
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()
)

if cotacao is None:
    raise ValueError("Não há cotação válida na tabela Bronze.")

taxa_dolar = cotacao["cotacaoCompra"]
print(f"Cotação utilizada: R$ {taxa_dolar} por dólar")

# 2. Leitura da Bronze
df_bronze_financeiro_filmes = spark.table(
    f"{bronze_schema}.tb_movies_financials"
)

df_silver_financeiro_filmes = (
    df_bronze_financeiro_filmes
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
)

# 3. Limpeza de orçamento e receita
for coluna in ["budget", "revenue"]:
    df_silver_financeiro_filmes = (
        df_silver_financeiro_filmes
        .withColumn(coluna, F.trim(F.col(coluna).cast("string")))

        # Ausência de dados vira NULL antes da conversão
        .withColumn(
            coluna,
            F.when(
                F.lower(F.col(coluna)).isin(
                    "", "unknown", "não informado", "nao informado",
                    "n/a", "na", "null", "none", "nan", "-", "nenhum"
                ),
                F.lit(None).cast("string")
            ).otherwise(F.col(coluna))
        )

        # Remove símbolos de moeda e espaços
        .withColumn(
            coluna,
            F.upper(
                F.regexp_replace(
                    F.col(coluna), r"(?i)USD|US\$|[$€£]|\s", ""
                )
            )
        )

        # Remove separadores de milhar e padroniza o decimal
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).rlike(r"^-?\d{1,3}(\.\d{3})+(,\d+)?$"),
                F.regexp_replace(
                    F.regexp_replace(F.col(coluna), r"\.", ""), ",", "."
                )
            )
            .when(
                F.col(coluna).rlike(r"^-?\d{1,3}(,\d{3})+(\.\d+)?$"),
                F.regexp_replace(F.col(coluna), ",", "")
            )
            .otherwise(F.regexp_replace(F.col(coluna), ",", "."))
        )

        # Converte para decimal, incluindo M = milhão e K = mil
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).endswith("M"),
                F.expr(
                    f"try_cast(regexp_replace({coluna}, 'M$', '') AS DECIMAL(18,6))"
                ) * 1000000
            )
            .when(
                F.col(coluna).endswith("K"),
                F.expr(
                    f"try_cast(regexp_replace({coluna}, 'K$', '') AS DECIMAL(18,6))"
                ) * 1000
            )
            .otherwise(F.expr(f"try_cast({coluna} AS DECIMAL(18,6))"))
            .cast("decimal(18,2)")
        )

        # Valores zerados ou negativos viram NULL
        .withColumn(
            coluna,
            F.when(F.col(coluna) > 0, F.col(coluna))
        )
    )

# 4. Mapeamento e cálculos
df_silver_financeiro_filmes = (
    df_silver_financeiro_filmes
    .select("id", "budget", "revenue")

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(taxa_dolar)).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(taxa_dolar)).cast("decimal(18,2)")
    )

    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)")
    )

    # Margem = lucro / receita × 100
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd") > 0,
            (F.col("lucro_usd") / F.col("receita_usd")) * 100
        ).cast("decimal(18,2)")
    )
)

# 5. Checagens de qualidade
id_valido = dq_check(
    "silver.tb_financeiro_filmes", "id_filme não nulo",
    df_silver_financeiro_filmes,
    F.col("id_filme").isNotNull()
)

orcamento_valido = dq_check(
    "silver.tb_financeiro_filmes", "orçamento positivo ou NULL",
    df_silver_financeiro_filmes,
    F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0)
)

receita_valida = dq_check(
    "silver.tb_financeiro_filmes", "receita positiva ou NULL",
    df_silver_financeiro_filmes,
    F.col("receita_usd").isNull() | (F.col("receita_usd") > 0)
)

if not (id_valido and orcamento_valido and receita_valida):
    raise ValueError("Há falhas nas checagens. Confira antes de gravar.")

display(df_silver_financeiro_filmes.limit(10))
df_silver_financeiro_filmes.printSchema()

Cotação utilizada: R$ 4.992900 por dólar
[PASS] silver.tb_financeiro_filmes | id_filme não nulo | 0/955485 linhas falharam
[PASS] silver.tb_financeiro_filmes | orçamento positivo ou NULL | 0/955485 linhas falharam
[PASS] silver.tb_financeiro_filmes | receita positiva ou NULL | 0/955485 linhas falharam


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,289588200.00,null,null,null,null
299536,300000000.00,2052415039.00,1497870000.00,10247503048.22,1752415039.00,8749633048.22,85.38
299534,356000000.00,2800000000.00,1777472400.00,13980120000.00,2444000000.00,12202647600.00,87.29
475557,55000000.00,1074458282.00,274609500.00,5364662756.20,1019458282.00,5090053256.20,94.88
271110,250000000.00,null,1248225000.00,null,null,null,null
284054,200000000.00,1349926083.00,998580000.00,6740045939.81,1149926083.00,5741465939.81,85.18
284052,180000000.00,676343174.00,898722000.00,3376913833.46,496343174.00,2478191833.46,73.39
315635,175000000.00,880166924.00,873757500.00,4394585434.84,705166924.00,3520827934.84,80.12
283995,200000000.00,863756051.00,998580000.00,4312647587.04,663756051.00,3314067587.04,76.85
297761,175000000.00,746846894.00,873757500.00,3728931857.05,571846894.00,2855174357.05,76.57


root
 |-- id_filme: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(18,2) (nullable = true)



## 2. silver.tb_financeiro_filmes — Gravação

Grava orçamento, receita, lucro e margem tratados na tabela financeira Silver, em Delta Overwrite.


In [0]:
# Grava os valores financeiros e as colunas calculadas.

df_silver_financeiro_filmes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

## 3. silver.tb_metricas_engajamento — Limpeza e limites

Padroniza a pontuação da popularidade antes da conversão. Usa conversão segura para notas e votos. Notas fora de 0 a 10 e valores negativos de popularidade ou contagem ficam NULL. A checagem temporária identifica possíveis perdas de números na conversão.


In [0]:
# As checagens validam a conversão e os limites numéricos definidos na atividade.

from pyspark.sql import functions as F

df_bronze_metricas_engajamento = spark.table(
    f"{bronze_schema}.tb_movies_metrics"
)

# 1. Limpeza da popularidade
df_silver_metricas_engajamento = (
    df_bronze_metricas_engajamento
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))

    # Remove espaços, aspas e barras de escape
    .withColumn(
        "popularity",
        F.regexp_replace(
            F.trim(F.col("popularity").cast("string")),
            r'[\s"\\]', ""
        )
    )

    # Identifica números para conferir possíveis perdas na conversão
    .withColumn(
        "_popularidade_numerica",
        F.coalesce(
            F.col("popularity").rlike(r"^\+?(?=.*\d)[\d.,]+$"),
            F.lit(False)
        )
    )

    # Padroniza separadores sem transformar 54,522 em 54522
    .withColumn(
        "popularity",
        F.when(
            F.col("popularity").rlike(r"^\+?\d{1,3}(\.\d{3})+,\d+$"),
            F.regexp_replace(
                F.regexp_replace(F.col("popularity"), r"\.", ""), ",", "."
            )
        )
        .when(
            F.col("popularity").rlike(r"^\+?\d{1,3}(,\d{3})+\.\d+$"),
            F.regexp_replace(F.col("popularity"), ",", "")
        )
        .otherwise(F.regexp_replace(F.col("popularity"), ",", "."))
    )

    .withColumn(
        "popularity",
        F.expr("try_cast(popularity AS DECIMAL(18,6))")
    )

    # Popularidade negativa vira NULL; zero permanece válido
    .withColumn(
        "popularity",
        F.when(F.col("popularity") >= 0, F.col("popularity"))
    )
)

# 2. Conversão segura das notas e validação da escala
for coluna in ["vote_average", "averageRating"]:
    df_silver_metricas_engajamento = (
        df_silver_metricas_engajamento
        .withColumn(
            coluna,
            F.expr(f"try_cast({coluna} AS DECIMAL(18,6))")
        )
        .withColumn(
            coluna,
            F.when(F.col(coluna).between(0, 10), F.col(coluna))
        )
    )

# 3. Conversão segura das contagens de votos
for coluna in ["vote_count", "numVotes"]:
    df_silver_metricas_engajamento = (
        df_silver_metricas_engajamento
        .withColumn(coluna, F.trim(F.col(coluna).cast("string")))

        # Aceita inteiros e valores como 100.0, sem truncar 100.5
        .withColumn(
            coluna,
            F.when(
                F.col(coluna).rlike(r"^[+-]?\d+(\.0+)?$"),
                F.expr(
                    f"try_cast(try_cast({coluna} AS DECIMAL(20,0)) AS BIGINT)"
                )
            )
        )

        # Contagem negativa vira NULL
        .withColumn(
            coluna,
            F.when(F.col(coluna) >= 0, F.col(coluna))
        )
    )

# 4. Mapeamento das colunas
df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .select(
        "id", "popularity", "vote_average", "vote_count",
        "averageRating", "numVotes", "_popularidade_numerica"
    )
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# 5. Checagens de qualidade
validacoes = [
    dq_check(
        "silver.tb_metricas_engajamento", "id_filme não nulo",
        df_silver_metricas_engajamento,
        F.col("id_filme").isNotNull()
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "popularidade numérica foi convertida",
        df_silver_metricas_engajamento,
        ~F.col("_popularidade_numerica") | F.col("popularidade").isNotNull()
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "nota TMDB entre 0 e 10 ou NULL",
        df_silver_metricas_engajamento,
        F.col("nota_media_tmdb").isNull()
        | F.col("nota_media_tmdb").between(0, 10)
    ),

    dq_check(
        "silver.tb_metricas_engajamento", "nota IMDb entre 0 e 10 ou NULL",
        df_silver_metricas_engajamento,
        F.col("nota_media_imdb").isNull()
        | F.col("nota_media_imdb").between(0, 10)
    )
]

for coluna in ["popularidade", "qtd_votos_tmdb", "qtd_votos_imdb"]:
    validacoes.append(
        dq_check(
            "silver.tb_metricas_engajamento",
            f"{coluna} não negativo ou NULL",
            df_silver_metricas_engajamento,
            F.col(coluna).isNull() | (F.col(coluna) >= 0)
        )
    )

if not all(validacoes):
    raise ValueError("Há falhas nas checagens. Confira antes de gravar.")

# Remove a coluna temporária da checagem
df_silver_metricas_engajamento = (
    df_silver_metricas_engajamento
    .drop("_popularidade_numerica")
)

display(df_silver_metricas_engajamento.limit(10))
df_silver_metricas_engajamento.printSchema()

[PASS] silver.tb_metricas_engajamento | id_filme não nulo | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade numérica foi convertida | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota TMDB entre 0 e 10 ou NULL | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | nota IMDb entre 0 e 10 ou NULL | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | popularidade não negativo ou NULL | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_tmdb não negativo ou NULL | 0/966276 linhas falharam
[PASS] silver.tb_metricas_engajamento | qtd_votos_imdb não negativo ou NULL | 0/966276 linhas falharam


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735000,7.606000,28894,8.000000,1270339
299536,154.340000,8.255000,27713,8.400000,1406782
299534,91.756000,8.263000,23857,8.400000,1484150
475557,54.522000,8.168000,23425,8.300000,1723035
271110,70.741000,7.400000,21541,7.800000,947222
284054,43.665000,7.390000,null,7.300000,924922
284052,70.535000,7.427000,20935,7.500000,895880
315635,65.880000,7.345000,20507,7.400000,835116
283995,67.553000,7.624000,20353,7.600000,844767
297761,35.356000,5.909000,20097,5.900000,null


root
 |-- id_filme: long (nullable = true)
 |-- popularidade: decimal(18,6) (nullable = true)
 |-- nota_media_tmdb: decimal(18,6) (nullable = true)
 |-- qtd_votos_tmdb: long (nullable = true)
 |-- nota_media_imdb: decimal(18,6) (nullable = true)
 |-- qtd_votos_imdb: long (nullable = true)



## 3. silver.tb_metricas_engajamento — Gravação

Grava as métricas de engajamento em Delta Overwrite. A coluna temporária de checagem já foi retirada do resultado.


In [0]:
# Grava apenas as colunas finais de métricas, com nomes em português.

df_silver_metricas_engajamento.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

## 4. silver.tb_avaliacoes_usuarios — Tratamento das avaliações

Remove duplicatas idênticas por filme, usuário, nota e comentário antes das conversões. Notas fora de 0 a 10 ficam NULL. Comentários ausentes ou formados apenas por espaços recebem Sem comentário.


In [0]:
# A deduplicação considera a combinação original das quatro colunas da avaliação.

from pyspark.sql import functions as F

df_bronze_avaliacoes_usuarios = spark.table(
    f"{bronze_schema}.tb_movies_reviews"
)

# Remove somente avaliações idênticas na origem
df_silver_avaliacoes_usuarios = (
    df_bronze_avaliacoes_usuarios
    .select("id", "nome", "nota", "comentario")
    .dropDuplicates(["id", "nome", "nota", "comentario"])
)

# Confere a deduplicação antes das conversões
dq_check_unique(
    "silver.tb_avaliacoes_usuarios",
    "sem avaliações idênticas na origem",
    df_silver_avaliacoes_usuarios,
    ["id", "nome", "nota", "comentario"]
)

df_silver_avaliacoes_usuarios = (
    df_silver_avaliacoes_usuarios

    # Tipagem
    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))
    .withColumn("nota", F.expr("try_cast(nota AS DOUBLE)"))

    # Notas fora de 0 a 10 ficam NULL
    .withColumn(
        "nota",
        F.when(F.col("nota").between(0, 10), F.col("nota"))
    )

    # Preenche comentários ausentes ou compostos apenas por espaços
    .withColumn(
        "comentario",
        F.when(
            F.col("comentario").isNull()
            | F.col("comentario").rlike(r"^\s*$"),
            "Sem comentário"
        ).otherwise(F.col("comentario"))
    )

    # Mapeamento das colunas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

# Checagens dos tratamentos
dq_check(
    "silver.tb_avaliacoes_usuarios",
    "nota entre 0 e 10 ou NULL",
    df_silver_avaliacoes_usuarios,
    F.col("nota_usuario").isNull()
    | F.col("nota_usuario").between(0, 10)
)

dq_check(
    "silver.tb_avaliacoes_usuarios",
    "comentário preenchido",
    df_silver_avaliacoes_usuarios,
    F.col("comentario_usuario").isNotNull()
    & ~F.col("comentario_usuario").rlike(r"^\s*$")
)

display(df_silver_avaliacoes_usuarios.limit(10))
df_silver_avaliacoes_usuarios.printSchema()

[PASS] silver.tb_avaliacoes_usuarios | sem avaliações idênticas na origem | 0 chaves duplicadas
[PASS] silver.tb_avaliacoes_usuarios | nota entre 0 e 10 ou NULL | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | comentário preenchido | 0/32412 linhas falharam


id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.
750569,Pedro Souza 611,4.4,Sem comentário
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.
823195,Rodrigo Mendes,2.9,Péssimo em todos os sentidos.
1055332,Gustavo Dias 303,null,Sem comentário
679864,Lucas Freitas 371,9.2,Perfeito! Um dos melhores filmes que já vi.


root
 |-- id_filme: long (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)



## 4. silver.tb_avaliacoes_usuarios — Gravação

Grava as avaliações tratadas em Delta Overwrite na tabela Silver.


In [0]:
# Grava as avaliações com as colunas renomeadas e os tratamentos aplicados.

# Gravação na Silver
df_silver_avaliacoes_usuarios.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

## 5. silver.tb_generos — Desmembramento e limpeza

Padroniza vírgula, ponto e vírgula e barra vertical antes do split. Explode cria uma linha por gênero do filme. Remove ruídos nas extremidades e conserva os valores da lista de gêneros válidos.


In [0]:
# A lista de gêneros válidos permite excluir números e textos deslocados desse campo.

generos_validos = [
    "action", "adventure", "animation", "comedy", "crime",
    "documentary", "drama", "family", "fantasy", "history",
    "horror", "music", "mystery", "romance", "science fiction",
    "tv movie", "thriller", "war", "western"
]

df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

df_silver_generos = (
    df_bronze_creditos
    .select("id", "genres")

    .withColumn("id", F.expr("try_cast(id AS BIGINT)"))

    # Padroniza os separadores antes do split
    .withColumn(
        "genres",
        F.regexp_replace(F.col("genres"), r"[;|]", ",")
    )

    # Separa os elementos e cria uma linha para cada gênero
    .withColumn(
        "genero",
        F.explode(F.split(F.col("genres"), ","))
    )

    # Remove espaços, aspas e barras residuais nas extremidades
    .withColumn(
        "genero",
        F.trim(
            F.regexp_replace(
                F.col("genero"),
                r'^[\s"\\]+|[\s"\\]+$',
                ""
            )
        )
    )

    # Mantém somente valores pertencentes ao domínio de gêneros
    .filter(F.lower(F.col("genero")).isin(generos_validos))

    .select("id", "genero")
    .withColumnRenamed("id", "id_filme")
)

# Checagem
dq_check(
    "silver.tb_generos",
    "gênero pertence ao domínio permitido",
    df_silver_generos,
    F.lower(F.col("genero")).isin(generos_validos)
)

display(df_silver_generos.limit(10))
df_silver_generos.printSchema()

[PASS] silver.tb_generos | gênero pertence ao domínio permitido | 0/1393272 linhas falharam


id_filme,genero
293660,Action
293660,Adventure
293660,Comedy
299536,Adventure
299536,Action
299536,Science Fiction
299534,Adventure
299534,Science Fiction
299534,Action
475557,Crime


root
 |-- id_filme: long (nullable = true)
 |-- genero: string (nullable = false)



## 5. silver.tb_generos — Gravação

Grava as associações entre id_filme e genero em Delta Overwrite.


In [0]:
# Grava os gêneros desmembrados, com um gênero em cada linha.

# Gravação na Silver
df_silver_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")


## 6. silver.tb_pessoas_empresas — Consolidação das entidades

Desmembra cast, directors, writers e production_companies. Identifica cada registro como Ator, Diretor, Roteirista ou Produtora, une as quatro origens, padroniza a capitalização e remove elementos vazios. Deduplica a combinação de filme, nome e tipo de entidade.


In [0]:
# A combinação de filme, nome e tipo preserva atuações diferentes da mesma pessoa.

from pyspark.sql import functions as F

df_bronze_creditos = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

# Padroniza os separadores das listas
df_creditos = df_bronze_creditos.withColumn(
    "id", F.expr("try_cast(id AS BIGINT)")
)

for coluna in ["cast", "directors", "writers", "production_companies"]:
    df_creditos = df_creditos.withColumn(
        coluna,
        F.regexp_replace(F.col(coluna), r"[;|]", ",")
    )

# Atores
df_atores = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("cast"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Ator"))
)

# Diretores
df_diretores = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("directors"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Diretor"))
)

# Roteiristas
df_roteiristas = (
    df_creditos
    .select(
        "id",
        F.explode(F.split(F.col("writers"), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Roteirista"))
)

# Produtoras
df_produtoras = (
    df_creditos
    .select(
        "id",
        F.explode(
            F.split(F.col("production_companies"), ",")
        ).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Produtora"))
)

# Consolidação e padronização
df_silver_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)

    .withColumn(
        "nome_entidade",
        F.initcap(F.lower(F.trim(F.col("nome_entidade"))))
    )

    # Remove elementos vazios gerados pelas listas
    .filter(
        F.col("nome_entidade").isNotNull()
        & ~F.col("nome_entidade").rlike(r"^\s*$")
    )

    .withColumnRenamed("id", "id_filme")

    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

df_silver_pessoas_empresas = (
    df_silver_pessoas_empresas

    # Remove URLs e nomes de arquivos de imagem
    .filter(
        ~F.col("nome_entidade").rlike(
            r"(?i)(^https?://|^www\.|\.(jpg|jpeg|png|webp|gif)(\?.*)?$)"
        )
    )

    # Um número sozinho não representa o nome de uma pessoa
    .filter(
        ~(
            F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
            & F.col("nome_entidade").rlike(r"^[+-]?\d+([.,]\d+)?$")
        )
    )
)

# Checagens
dq_check_unique(
    "silver.tb_pessoas_empresas",
    "associações sem duplicatas",
    df_silver_pessoas_empresas,
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

dq_check(
    "silver.tb_pessoas_empresas",
    "tipo de entidade válido",
    df_silver_pessoas_empresas,
    F.col("tipo_entidade").isin(
        "Ator", "Diretor", "Roteirista", "Produtora"
    )
)

dq_check(
    "silver.tb_pessoas_empresas",
    "nome da entidade preenchido",
    df_silver_pessoas_empresas,
    F.col("nome_entidade").isNotNull()
    & ~F.col("nome_entidade").rlike(r"^\s*$")
)

display(df_silver_pessoas_empresas.limit(10))
df_silver_pessoas_empresas.printSchema()

[PASS] silver.tb_pessoas_empresas | associações sem duplicatas | 0 chaves duplicadas
[PASS] silver.tb_pessoas_empresas | tipo de entidade válido | 0/915186 linhas falharam
[PASS] silver.tb_pessoas_empresas | nome da entidade preenchido | 0/915186 linhas falharam


id_filme,nome_entidade,tipo_entidade
263115,Stephen Dunlevy,Ator
374720,Mark Rylance,Ator
453395,Benedict Cumberbatch,Ator
328387,Brian Marc,Ator
334533,Frank Langella,Ator
331482,Tracy Letts,Ator
460465,Joe Taslim,Ator
481848,Karen Gillan,Ator
531309,Meredith Hagner,Ator
127380,Eugene Levy,Ator


root
 |-- id_filme: long (nullable = true)
 |-- nome_entidade: string (nullable = false)
 |-- tipo_entidade: string (nullable = false)



## 6. silver.tb_pessoas_empresas — Gravação

Grava as associações de pessoas e empresas em uma única tabela Silver, em Delta Overwrite.


In [0]:
# Grava a tabela unificada com os quatro tipos de entidade.

# Gravação na Silver
df_silver_pessoas_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

## 7. silver.tb_cotacao_dolar — Série contínua e Forward Fill

Cria todos os dias entre a primeira e a última data presentes na origem. Para cada dia, seleciona a cotação válida mais recente até aquela data. Assim, dias sem cotação recebem o último valor disponível.


In [0]:
# O preenchimento usa a última cotação disponível e cobre o intervalo da origem.

from pyspark.sql import functions as F

df_bronze_cotacao_dolar = spark.table(
    f"{bronze_schema}.tb_cotacao_dolar"
)

# Tipagem e nomes em português
df_cotacoes = (
    df_bronze_cotacao_dolar
    .select(
        F.expr(
            "try_cast(dataHoraCotacao AS TIMESTAMP)"
        ).alias("data_hora_cotacao"),

        F.expr(
            "try_cast(cotacaoCompra AS DECIMAL(18,6))"
        ).alias("cotacao_compra")
    )
)

# Cria uma linha para cada dia do histórico
df_calendario = (
    df_cotacoes
    .agg(
        F.min(F.to_date("data_hora_cotacao")).alias("data_inicio"),
        F.max(F.to_date("data_hora_cotacao")).alias("data_fim")
    )
    .select(
        F.explode(
            F.sequence(
                F.col("data_inicio"),
                F.col("data_fim"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

# Cotações disponíveis para o preenchimento
df_cotacoes_disponiveis = df_cotacoes.filter(
    F.col("data_hora_cotacao").isNotNull()
    & F.col("cotacao_compra").isNotNull()
)

# Forward Fill: última cotação disponível até cada dia
df_silver_cotacao_dolar = (
    df_calendario
    .join(
        df_cotacoes_disponiveis,
        F.to_date(df_cotacoes_disponiveis["data_hora_cotacao"])
        <= df_calendario["data_cotacao"],
        how="left"
    )
    .groupBy("data_cotacao")
    .agg(
        F.max(
            F.struct("data_hora_cotacao", "cotacao_compra")
        ).alias("ultima_cotacao")
    )
    .select(
        "data_cotacao",
        F.col("ultima_cotacao.cotacao_compra").alias("cotacao_compra")
    )
)

# Checagens
dq_check_unique(
    "silver.tb_cotacao_dolar",
    "uma cotação por dia",
    df_silver_cotacao_dolar,
    ["data_cotacao"]
)

dq_check(
    "silver.tb_cotacao_dolar",
    "dias do histórico com cotação preenchida",
    df_silver_cotacao_dolar,
    F.col("cotacao_compra").isNotNull()
)

display(df_silver_cotacao_dolar.orderBy("data_cotacao"))
df_silver_cotacao_dolar.printSchema()

[PASS] silver.tb_cotacao_dolar | uma cotação por dia | 0 chaves duplicadas
[PASS] silver.tb_cotacao_dolar | dias do histórico com cotação preenchida | 0/7 linhas falharam


data_cotacao,cotacao_compra
2026-10-01,5.207300
2026-10-02,5.223200
2026-10-03,5.223200
2026-10-04,5.223200
2026-10-05,4.985300
2026-10-06,4.969200
2026-10-07,4.992900


root
 |-- data_cotacao: date (nullable = false)
 |-- cotacao_compra: decimal(18,6) (nullable = true)



## 7. silver.tb_cotacao_dolar — Gravação

Grava o histórico diário preenchido em Delta Overwrite, com data_cotacao e cotacao_compra.


In [0]:
# Grava a série diária resultante do Forward Fill.

# Gravação na Silver
df_silver_cotacao_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")